Made by Jose Alan Barraza Villaverde

# PLM Control Guide
Complete workflow for controlling the DLP670S PLM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PLMController import PLMController
import time
from scipy.ndimage import zoom
import ctypes

## 1. Create Controller Instance
Adjust `MAX_FRAMES`, `N` (width), `M` (height), and the PLM monitor position `(x0, y0)` to match your setup.

In [ ]:
MAX_FRAMES = 2
N = 1358   # PLM width
M = 800    # PLM height
# x0 and y0 control where the menu appears.
x0 = 1920
y0 = 0

dll_path = r'..\bin\plmctrl.dll'

plm = PLMController(MAX_FRAMES, N, M, dll_path, x0, y0)

## 2. Open USB Connection
Must be called **before** any configuration or display commands.

In [ ]:
res = plm.open()

## 3. Configure the PLM
The `configure()` method runs the full setup sequence with proper delays:
- Set source to Parallel RGB (24-bit)
- Set port swap (ABC → ABC)
- Set connection type (HDMI or DisplayPort)
- Set video pattern mode
- Update lookup table

In [ ]:
source = 0 # Parallel RGB
port_width = 1
plm.set_source(source, port_width)
# plm.set_port_swap(0,0)
plm.set_port_swap(1,0) # Only use when working with Display Port
connection_type = 2 # 1 = HDMI, 2 = Display Port.
plm.set_pixel_mode(connection_type)
time.sleep(5) # It's important to pause between commands to setup the connection.
plm.set_connection_type(connection_type)
time.sleep(10)
plm.set_video_pattern_mode()
play_mode = 1 # 0 = Play Once, 1 = Continuous (Repeat mode)
plm.update_lut(play_mode, connection_type)

## 3. Start UI (Window)
`windowed=True` is useful for testing – it shows the PLM output in a resizable window.

In [ ]:
plm.set_windowed(True)
plm.start_ui()

## 5. Set Phase Map & Lookup Table
These define how the PLM maps grayscale values to phase levels.

- Phase Levels: Is a 17-element lookup table that maps continuous phase values (normalized 0 to 1, representing 0 to 2π) to 16 discrete quantization levels (0-15) for the PLM hardware plmctrl.cpp:157. It is used during the quantization step to determine which discrete level a continuous phase value belongs to plmctrl.cpp:931-941.

- Phase Map: Is a 16×4 lookup table (64 integers total) that maps each quantized phase level (0-15) to the specific binary state of the 4 sub-pixels that compose a single PLM logical pixel plmctrl.cpp:160-177. Each entry determines which bits are set for each sub-pixel during the bitpacking process plmctrl.cpp:972-975.

In [ ]:
# Phase levels (voltage levels for the micromirrors)
# My calibration
phase_levels = np.array((0.04360591289392656, 0.044775494427230215, 0.06471969209310459, 0.08334685080484126, 
                         0.08336777638103532, 0.08383399845364213, 0.2925851213252165, 0.4442366203039024, 
                         0.5817363154023534, 0.6331495384760347, 0.7226563427330012, 0.7896310130733268,
                         0.8297999579658473, 0.842104990522558, 0.9334540476741956, 0.9984680371258492), dtype=np.float32)
phase_map_order = (6, 0, 14, 4, 12, 8, 3, 11, 7, 15, 1, 9, 5, 13, 2, 10)
# TI/Jose's Calibration
phase_levels = np.array((0.004, 0.017, 0.036, 0.058, 0.085, 0.117, 0.157, 0.217, 0.296, 0.4, 0.5, 0.605, 0.713, 0.82, 0.922, 0.981, 1), dtype=np.float32)
phase_map_order = (12, 8, 4, 14, 0, 6, 10, 2, 13, 5, 9, 1, 15, 7, 11, 3)

plm.set_lookup_table(phase_levels)

# Phase map (which mirrors/voltages are active for each level)
phase_map = np.array([
    [0,0,0,0], [1,0,0,0], [0,1,0,0], [1,1,0,0],
    [0,0,1,0], [1,0,1,0], [0,1,1,0], [1,1,1,0],
    [0,0,0,1], [1,0,0,1], [0,1,0,1], [1,1,0,1],
    [0,0,1,1], [1,0,1,1], [0,1,1,1], [1,1,1,1],
], dtype=np.int32)



# This order of the states from the calibration.
phase_map = phase_map[phase_map_order, :]

plm.set_phase_map(phase_map)

## 6. Generate & Bitpack Holograms

### Method A: Bitpack and insert one frame at a time (simplest)

In [ ]:
import numpy as np
from hologram_generator import gerchberg_saxton, load_and_prepare_target

def get_phase_for_gpu(image_path, iterations=20, seed=42):
    """Returns phase as 3D float32 array in [0, 1] for bitpack_holograms_gpu."""
    cgh_size = (1358, 800)
    half_size = (cgh_size[0] // 2, cgh_size[1] // 2)
    small_target = load_and_prepare_target(image_path, half_size)
    
    target = np.zeros((cgh_size[1], cgh_size[0]), dtype=np.float64)
    sh, sw = half_size[1], half_size[0]
    
    # Calculate center position
    start_row = (cgh_size[1] - sh) // 2
    start_col = (cgh_size[0] - sw) // 2
    
    # Place small_target in the center
    target[start_row:start_row+sh, start_col:start_col+sw] = small_target

    # Get continuous phase in radians [-π, π]
    phase_rad = gerchberg_saxton(target, iterations=iterations, random_seed=seed)
    
    # Normalize to [0, 1] and add batch dimension (1, H, W)
    phase_normalized = (phase_rad + np.pi) / (2 * np.pi)
    return phase_normalized.astype(np.float32)[np.newaxis, ...]  # shape: (1, 800, 1358)

In [ ]:
phase = get_phase_for_gpu(r"C:\Users\Alan_\Downloads\Summer Internship\work\PLM_TI_stuff\Controller\plmctrl\python_wrapper\hologram_generator\pickle_rick.png")
phase = np.repeat(phase, 24, axis=0)
frame = plm.bitpack_holograms_gpu(phase)
plm.insert_frames(frame, offset=0, format=1)  # format 1 = RGBA

# Laguerre Gauss Holograms

In [ ]:
from phase_holograms.fields_propagation.fields import laguerre_gauss, hermite_gauss, speckle_gauss
from phase_holograms.holograms.phase_holograms import hologram_type1, hologram_type2, hologram_type3

nx = 1358 ; ny = 800 
X,Y = np.meshgrid(np.arange(nx)-nx/2,np.arange(ny)-ny/2)

Nt = 4 # Total order of LG beam
ell = 2 # topologicla charge
hlg_sc= 3*np.sqrt(Nt+1)/ny # Size of LG beam scales as Nt^(1/2)
field = laguerre_gauss(Nt, ell, hlg_sc*(X-0), hlg_sc*Y) +laguerre_gauss(3, 1, hlg_sc*(X-0), hlg_sc*Y)
# field = hermite_gauss(Nt, ell, hlg_sc*X, hlg_sc*Y)
nuvec = 0.1*np.array([1.,1.])#np.array([1/3,1/3])
holo = hologram_type3(field, nuvec, fa=None)
holo = (holo - holo.min()) / (holo.max() - holo.min())
phase = np.tile(holo, (24, 1, 1)).astype(np.float32)
frame = plm.bitpack_holograms_gpu(phase)
plm.insert_frames(frame, offset=0, format=1)

In [ ]:
plt.imshow(np.abs(field)**2)

# Speckle Pattern for Dynamic Complex Media

In [ ]:
phase = np.zeros((24, M, N), dtype=np.float32)

# rnd_phase = np.random.rand(M//2,N//2)/4 + .5
rnd_phase = np.random.normal(.5,.25,(M//4,N//4))
rnd_phase = zoom(rnd_phase,4, order=0)
rnd_phase = np.resize(rnd_phase, (M, N))
# rnd_phase = np.random.normal(.5,.2,(M,N))
rnd_phase[rnd_phase<0] = 0
rnd_phase[rnd_phase>1] = 1
phase[:] = rnd_phase[None,...]

print(phase)
print(phase.shape)
plt.imshow(phase[0,:,:])

frame = plm.bitpack_holograms_gpu(phase)

plm.insert_frames(frame, 0, format=1)

In [ ]:
for i in range(MAX_FRAMES):
    phase = np.zeros((24, M, N), dtype=np.float32)
    rnd_phase = np.random.normal(.5, .25, (M//4, N//4))
    rnd_phase = zoom(rnd_phase, 4, order=0)
    rnd_phase = np.resize(rnd_phase, (M, N))
    rnd_phase[rnd_phase<0] = 0
    rnd_phase[rnd_phase>1] = 1
    phase[:] = rnd_phase[None,...]

    frame = plm.bitpack_holograms_gpu(phase)
    plm.insert_frames(frame, i, format=1)

sequence = np.arange(MAX_FRAMES, dtype=np.uint64)
plm.set_frame_sequence(sequence)
plm.start_sequence(MAX_FRAMES)

In [ ]:
phase = np.zeros((24, M, N), dtype=np.float32)

rnd_phase1 = np.random.normal(.5, .15, (M//4, N//4))
rnd_phase1 = zoom(rnd_phase1, 4, order=0)
rnd_phase1 = np.resize(rnd_phase1, (M, N))
rnd_phase1[rnd_phase1<0] = 0
rnd_phase1[rnd_phase1>1] = 1

rnd_phase2 = np.random.normal(.5, .15, (M//4, N//4))
rnd_phase2 = zoom(rnd_phase2, 4, order=0)
rnd_phase2 = np.resize(rnd_phase2, (M, N))
rnd_phase2[rnd_phase2<0] = 0
rnd_phase2[rnd_phase2>1] = 1

rnd_phase3 = np.random.normal(.5, .15, (M//4, N//4))
rnd_phase3 = zoom(rnd_phase3, 4, order=0)
rnd_phase3 = np.resize(rnd_phase3, (M, N))
rnd_phase3[rnd_phase3<0] = 0
rnd_phase3[rnd_phase3>1] = 1

rnd_phase4 = np.random.normal(.5, .15, (M//4, N//4))
rnd_phase4 = zoom(rnd_phase4, 4, order=0)
rnd_phase4 = np.resize(rnd_phase4, (M, N))
rnd_phase4[rnd_phase4<0] = 0
rnd_phase4[rnd_phase4>1] = 1

In [ ]:
rnd_phase1 = np.load(r'4_complex\rnd_phase1.npy')
rnd_phase2 = np.load(r'4_complex\rnd_phase2.npy')
rnd_phase3 = np.load(r'4_complex\rnd_phase3.npy')
rnd_phase4 = np.load(r'4_complex\rnd_phase4.npy')
# phase = np.zeros((24, M, N), dtype=np.float32)
# phase[:] = rnd_phase1[None,...]
# frame = plm.bitpack_holograms_gpu(phase)

# plm.insert_frames(frame, 0, format=1)

In [55]:
numHolograms = 24
MAX_FRAMES = 1
frame = np.zeros((MAX_FRAMES, 2*M, 4*2*N), dtype=np.uint8)
phase = np.zeros((numHolograms, M, N), dtype=np.float32)

rnd_phases = [rnd_phase4, rnd_phase3, rnd_phase2, rnd_phase1]

for i in range(numHolograms):
    phase[i] = rnd_phases[i % 4][None, ...]

phase_ptr = phase.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
frame_ptr = frame[:, :, :].ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))
plm.bitpack_holograms_gpu_ptr(phase_ptr, frame_ptr, numHolograms)

plm.insert_frames(frame, 0, format=1)

1

In [ ]:
numHolograms = 24
MAX_FRAMES = 1
frame = np.zeros((MAX_FRAMES, 2*M, 4*2*N), dtype=np.uint8)
phase = np.zeros((numHolograms, M, N), dtype=np.float32)

for j in range(numHolograms):
    phase[j] = (rnd_phase1 if j % 2 == 0 else rnd_phase2)[None, ...]

phase_ptr = phase.ctypes.data_as(ctypes.POINTER(ctypes.c_float))
frame_ptr = frame[:, :, :].ctypes.data_as(ctypes.POINTER(ctypes.c_uint8))
plm.bitpack_holograms_gpu_ptr(phase_ptr, frame_ptr, numHolograms)

plm.insert_frames(frame, 0, format=1)

In [ ]:
np.save('rnd_phase1.npy', rnd_phase1)
np.save('rnd_phase2.npy', rnd_phase2)
np.save('rnd_phase3.npy', rnd_phase3)
np.save('rnd_phase4.npy', rnd_phase4)

In [53]:
for i in range(MAX_FRAMES):
    phase = np.zeros((24, M, N), dtype=np.float32)
    rnd_phase = np.random.rand(M, N) / 2.3
    phase[:] = rnd_phase1[None, ...]
    
    frame = plm.bitpack_holograms_gpu(phase)
    plm.insert_frames(frame, i, format=1)

# 4. Set sequence and play
sequence = np.arange(MAX_FRAMES, dtype=np.uint64)
plm.set_frame_sequence(sequence)
plm.start_sequence(MAX_FRAMES)

### Method B: Bitpack and insert all at once (faster)

In [ ]:
phase = np.zeros((M,N), dtype=np.float32)
phase[:, N//2:] = phase_levels[0]
phase = np.tile(phase[np.newaxis, :, :], (24,1,1))
print(phase.shape)
print(phase)

# Bitpack and insert
frame = plm.bitpack_holograms_gpu(phase)  
plm.insert_frames(frame, 0, format=1)


## 7. Set Frame Sequence and Play

The frame sequence defines what order the inserted frames play in.

In [ ]:
# Sequence of frame indices to play (length must equal MAX_FRAMES)
sequence = np.arange(MAX_FRAMES, dtype=np.uint64)
plm.set_frame_sequence(sequence)

In [ ]:
# Start playback
plm.start_sequence(MAX_FRAMES)
print("Playing sequence - call plm.stop() to stop")

### Manual play/stop (alternative to start_sequence)

In [ ]:
plm.play()

In [ ]:
plm.stop()

## 8. Cleanup
Always run these cells to properly disconnect and release resources.

In [ ]:
plm.stop()
plm.stop_ui()
plm.lib.Close()
print("Cleanup complete")

## Quick Reference

| Step | Method | Notes |
|------|--------|-------|
| Init | `PLMController(MAX_FRAMES, N, M, dll_path, x0, y0)` | |
| Window | `set_windowed(True)` then `start_ui()` | Windowed for testing |
| Connect | `open()` | Must come first |
| Config | `configure(play_mode, conn_type)` | Once per power cycle |
| Phase LUT | `set_lookup_table(levels)` | `float32` array |
| Phase map | `set_phase_map(map)` | `int32` 2D array |
| Bitpack | `bitpack_holograms_gpu(phase)` → frame | `float32` → `uint8` |
| Insert | `insert_frames(frame, offset, format)` | `format=1` for RGBA |
| Sequence | `set_frame_sequence(seq)` | `uint64` array |
| Play | `start_sequence(n)` or `play()` | |
| Stop | `stop()` | |
| Cleanup | `stop_ui()` then `lib.Close()` | |